In [ ]:
import os

my_bucket = os.getenv('WORKSPACE_BUCKET')
my_bucket

In [ ]:
!gsutil ls $WORKSPACE_BUCKET/notebooks/scratch/2025-05-14_viral_disease_cohort_creation/

In [ ]:
#!gsutil ls $WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/

In [ ]:
#### import all raw cohorts PKLs

In [ ]:
def load_pkl_from_bucket(name_of_pkl):  
    # This snippet assumes you run setup first

    # This code copies file in your Google Bucket and loads it into a dataframe

    # Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
    name_of_file_in_bucket = name_of_pkl

    ########################################################################
    ##
    ################# DON'T CHANGE FROM HERE ###############################
    ##
    ########################################################################

    # get the bucket name
    my_bucket = os.getenv('WORKSPACE_BUCKET')

    # copy csv file from the bucket to the current working space
    os.system(f"gsutil cp '{my_bucket}/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/{name_of_file_in_bucket}' .")

    print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
    
    return name_of_file_in_bucket 

In [ ]:
#import all raw cohort PKLs

#control
load_pkl_from_bucket("controls.pkl")

load_pkl_from_bucket('controls_vax_data.pkl')

#NS cohorts
load_pkl_from_bucket('nonseasonal_cohort_data_table.pkl')

#NS vax cohorts
load_pkl_from_bucket('nonseasonal_vaccinated_cohort_data_table.pkl')

#covid
load_pkl_from_bucket('seasonal_vaccine_COVID_cohort_data_table.pkl')

#influenza
load_pkl_from_bucket('seasonal_vaccine_influenza_cohort_data_table.pkl')

In [ ]:
#import all binned cohort PKLs

#non_seasonal_cohort_binning
load_pkl_from_bucket('nonseasonal_B1_B2_data_table.pkl')


#non_seasonal_vaccinated_cohort_binning
load_pkl_from_bucket('nonseasonal_vaccinated_D1_D2_data_table.pkl')
load_pkl_from_bucket('nonseasonal_vaccinated_C1_C2_data_table.pkl')

#covid
load_pkl_from_bucket('seasonal_vaccine_COVID_strain_bin_cohort_data_table.pkl')

#influenza
load_pkl_from_bucket('seasonal_vaccine_influenza_seasonal_bin_cohort_data_table.pkll')


In [ ]:
#### load pkl data frames

In [ ]:

import pandas as pd
import pickle
from pathlib import Path

def load_pickled_data(file_path):
    """
    Loads and deserializes data from a specified pickle file.

    Args:
        file_path (str or Path): The path to the .pkl file.

    Returns:
        The deserialized data (e.g., DataFrame, dict, list)
        or None if an error occurs.
    """
    # Ensure the input is a Path object
    p = Path(file_path)
    
    with open(p, 'rb') as f:
        data = pickle.load(f)
        
    return data

In [ ]:
#import all raw cohort PKLs

#control
control = load_pickled_data("controls.pkl")

control_vax = load_pickled_data('controls_vax_data.pkl')

#NS cohorts
ns_df = load_pickled_data('nonseasonal_cohort_data_table.pkl')

#NS vax cohorts
ns_vax_df = load_pickled_data('nonseasonal_vaccinated_cohort_data_table.pkl')

#covid
covid = load_pickled_data('seasonal_vaccine_COVID_cohort_data_table.pkl')

#influenza
flu = load_pickled_data('seasonal_vaccine_influenza_cohort_data_table.pkl')

In [ ]:
#load binned pkl data frames

#non_seasonal_cohort_binning
ns_B = load_pickled_data('nonseasonal_B1_B2_data_table.pkl')


#non_seasonal_vaccinated_cohort_binning
ns_D = load_pickled_data('nonseasonal_vaccinated_D1_D2_data_table.pkl')
ns_C = load_pickled_data('nonseasonal_vaccinated_C1_C2_data_table.pkl')

#covid
covid_bin = load_pickled_data('seasonal_vaccine_COVID_strain_bin_cohort_data_table.pkl')

#influenza
flu_bin = load_pickled_data('seasonal_vaccine_influenza_seasonal_bin_cohort_data_table.pkll')


In [ ]:
#### wrangle data frames

In [ ]:
import pandas as pd
import numpy as np

def wrangle_control_cohort_data(df):
    
    # Make a copy to avoid modifying the original DataFrame
    wrangled_df = df.copy()

    # --- 1. Merge Race/Ethnicity ---
    
    # Define the helper function for merging
    def merge_race_ethnicity_data(row):
        """Helper function to apply row-wise."""
        if row["ethnicity"] == "Hispanic or Latino":
            return row["ethnicity"]
        else:
            return row["race"]

    # Apply the function to create the 'updated_race' column
    wrangled_df["updated_race"] = wrangled_df.apply(merge_race_ethnicity_data, axis=1)

    # --- 2. Filter Excluded Groups ---
    
    # List of races to exclude
    to_drop = [
        'American Indian or Alaska Native',
    ]

    # Keep only rows whose updated_race is NOT in the to_drop list
    wrangled_df = wrangled_df[~wrangled_df['updated_race'].isin(to_drop)]


    return wrangled_df

In [ ]:
#control

ctrl_upd = wrangle_control_cohort_data(control)

In [ ]:
import pandas as pd
import numpy as np

def wrangle_vax_cohort_data(df):
    """
    Wrangles the cohort DataFrame by merging race/ethnicity,
    filtering specific groups, and calculating age.
    
    Args:
        df (pd.DataFrame): The input DataFrame. Must contain 
                         'ethnicity', 'race', 'first_diag_date_x', 
                         and 'date_of_birth' columns.

    Returns:
        pd.DataFrame: The wrangled DataFrame.
    """
    
    # Make a copy to avoid modifying the original DataFrame
    wrangled_df = df.copy()

    # --- 2. Filter Excluded Groups ---
    
    # List of races to exclude
    to_drop = [
        'American Indian or Alaska Native',
    ]

    # Keep only rows whose updated_race is NOT in the to_drop list
    wrangled_df = wrangled_df[~wrangled_df['updated_race'].isin(to_drop)]

    # --- 3. Calculate Age ---
    
    # Ensure date columns are datetime objects, handling potential errors
    # Using .loc to avoid a potential SettingWithCopyWarning
    wrangled_df.loc[:, 'first_diag_date_x'] = pd.to_datetime(wrangled_df['first_diag_date_x'], errors="coerce")
    wrangled_df.loc[:, 'date_of_birth'] = pd.to_datetime(wrangled_df['date_of_birth'], errors="coerce")

    # Calculate age in years
    # This will result in NaN if either date was invalid (became NaT)
    wrangled_df['age'] = (wrangled_df['first_diag_date_x'] - wrangled_df['date_of_birth']).dt.days / 365.25

    return wrangled_df


In [ ]:
#ns vax cohorts

upd_ns_vax_cases = {}

for key, table in ns_vax_df.items():
    
    tmp = wrangle_vax_cohort_data(table)
    
    upd_ns_vax_cases[key] = tmp



In [ ]:
#flu and covid
#covid
covid_df = covid[(37311061,'COVID-19')]
covid_df_upd = wrangle_vax_cohort_data(covid_df)

#flu
flu_df = flu[(46273463, 'Upper respiratory tract infection due to Influenza')]
flu_df_upd = wrangle_vax_cohort_data(flu_df)    
    

In [ ]:
import pandas as pd
import numpy as np

def wrangle_ns_cohorts_data(df):
    
    
    # Make a copy to avoid modifying the original DataFrame
    wrangled_df = df.copy()


    # --- 2. Filter Excluded Groups ---
    
    # List of races to exclude
    to_drop = [
        'American Indian or Alaska Native',
    ]

    # Keep only rows whose updated_race is NOT in the to_drop list
    wrangled_df = wrangled_df[~wrangled_df['updated_race'].isin(to_drop)]

    # --- 3. Calculate Age ---
    
    # Ensure date columns are datetime objects, handling potential errors
    # Using .loc to avoid a potential SettingWithCopyWarning
    wrangled_df.loc[:, 'first_diag_date'] = pd.to_datetime(wrangled_df['first_diag_date'], errors="coerce")
    wrangled_df.loc[:, 'date_of_birth'] = pd.to_datetime(wrangled_df['date_of_birth'], errors="coerce")

    # Calculate age in years
    # This will result in NaN if either date was invalid (became NaT)
    wrangled_df['age'] = (wrangled_df['first_diag_date'] - wrangled_df['date_of_birth']).dt.days / 365.25

    return wrangled_df

In [ ]:
#NS cohorts
upd_ns_cases = {}

for key, table in ns_df.items():
    
    tmp = wrangle_ns_cohorts_data(table)
    
    upd_ns_cases[key] = tmp

In [ ]:
#### get control age

In [ ]:
import pandas as pd

def calculate_vax_cohorts_control_age_at_index_date(cases_df, output_csv_path, case_dx_date_col="first_diag_date_x"):  
    
    
    """
    Calculates control ages based on the median case diagnosis date (index date)
    and saves the resulting control DataFrame to a CSV.
    
    Args:
        cases_df (pd.DataFrame): DataFrame of the case cohort.
        
        output_csv_path (str): File path to save the new control CSV.
    
        case_dx_date_col (str): Column name for diagnosis dates in cases_df.
        
        
    Returns:
        pd.DataFrame: The modified control DataFrame with the new 'age' column.
    """
    
    
    controls_df = ctrl_upd
    id_cases_col="person_id"
    ctrl_dob_col="date_of_birth" 
    
    # Make copies to avoid modifying the original DataFrames
    cases = cases_df.copy()
    ctrls = controls_df.copy()

    # 1) First diagnosis per case, then the cohort-wide median of those dates
    cases[case_dx_date_col] = pd.to_datetime(cases[case_dx_date_col], errors="coerce")
    first_dx = (cases
                .dropna(subset=[case_dx_date_col])
                .sort_values(case_dx_date_col)
                .groupby(id_cases_col, as_index=False)[case_dx_date_col].first())

    index_date = first_dx[case_dx_date_col].median()  # single anchor date
    print(f"Index date (median first diagnosis): {index_date.date()}")

    # 2) Controls: age at the index date (from DOB)
    ctrls[ctrl_dob_col] = pd.to_datetime(ctrls[ctrl_dob_col], errors="coerce")
    ctrls["age"] = (index_date - ctrls[ctrl_dob_col]).dt.days / 365.25
    
    # 3) Save to CSV (matches the original snippet's lack of index=False)
    ctrls.to_csv(output_csv_path)
    
    print(f"Control cohort with ages saved to: {output_csv_path}")
    
    # 4) Return the DataFrame for further use
    return ctrls

In [ ]:
#covid and flu

#covid
covid_controls = calculate_vax_cohorts_control_age_at_index_date(covid_df_upd, 'cohort_data/covid_controls.csv', case_dx_date_col="first_diag_date_x")


#flu
flu_controls = calculate_vax_cohorts_control_age_at_index_date(flu_df_upd,'cohort_data/flu_controls.csv', case_dx_date_col="first_diag_date_x")


In [ ]:
import pandas as pd

def calculate_control_age_at_index_date(
    cases_df,
    controls_df,
    output_csv_path,
    id_cases_col="person_id",
    case_dx_date_col="first_diag_date",
    ctrl_dob_col="date_of_birth"
):
    
    """
    Calculates control ages based on the median case diagnosis date (index date)
    and saves the resulting control DataFrame to a CSV.
    
    Args:
        cases_df (pd.DataFrame): DataFrame of the case cohort.
        controls_df (pd.DataFrame): DataFrame of the control cohort.
        output_csv_path (str): File path to save the new control CSV.
        id_cases_col (str): Column name for case IDs in cases_df.
        case_dx_date_col (str): Column name for diagnosis dates in cases_df.
        ctrl_dob_col (str): Column name for date of birth in controls_df.
        
    Returns:
        pd.DataFrame: The modified control DataFrame with the new 'age' column.
    """
    
    
    
    
    # Make copies to avoid modifying the original DataFrames
    cases = cases_df.copy()
    ctrls = controls_df.copy()

    # 1) First diagnosis per case, then the cohort-wide median of those dates
    cases[case_dx_date_col] = pd.to_datetime(cases[case_dx_date_col], errors="coerce")
    first_dx = (cases
                .dropna(subset=[case_dx_date_col])
                .sort_values(case_dx_date_col)
                .groupby(id_cases_col, as_index=False)[case_dx_date_col].first())

    index_date = first_dx[case_dx_date_col].median()  # single anchor date
    print(f"Index date (median first diagnosis): {index_date.date()}")

    # 2) Controls: age at the index date (from DOB)
    ctrls[ctrl_dob_col] = pd.to_datetime(ctrls[ctrl_dob_col], errors="coerce")
    ctrls["age"] = (index_date - ctrls[ctrl_dob_col]).dt.days / 365.25

    # 3) Save to CSV (matches the original snippet's lack of index=False)
    ctrls.to_csv(output_csv_path)
    
    print(f"Control cohort with ages saved to: {output_csv_path}")
    
    # 4) Return the DataFrame for further use
    return ctrls

In [ ]:
#covid and flu


covid_ctrls = calculate_control_age_at_index_date(
                cases_df=covid_df_upd,
                controls_df=ctrl_upd, # Use the loaded master_control_df
                output_csv_path='ns_vax_controls/covid_controls.csv',
                id_cases_col="person_id",
                case_dx_date_col="first_diag_date_x",
                ctrl_dob_col="date_of_birth"
            )

flu_ctrls = calculate_control_age_at_index_date(
                cases_df=flu_df_upd,
                controls_df=ctrl_upd, # Use the loaded master_control_df
                output_csv_path='ns_vax_controls/flu_controls.csv',
                id_cases_col="person_id",
                case_dx_date_col="first_diag_date_x",
                ctrl_dob_col="date_of_birth"
            )



In [ ]:
#NS cohort - getting controls age
import pandas as pd
import os
import re
from pathlib import Path
import sys

def slugify(text):
    """
    Converts a string into a safe component for a filename.
    'Viral disease' -> 'viral_disease'
    'Acute hepatitis C' -> 'acute_hepatitis_c'
    '...AND/OR...' -> 'and_or'
    """
    text = str(text).lower()
    # Replace non-alphanumeric/underscore with a single underscore
    text = re.sub(r'[^a-z0-9_]+', '_', text) 
    text = text.strip('_') # Clean up leading/trailing underscores
    return text

def process_ns_cohorts(case_cohort_dict, 
                        control_df_path, 
                        output_folder="cohort_data_ns"):
    """
    Processes a dictionary of case cohorts against a single master control cohort.

    This function loads the master control file, then iterates through each
    case cohort, calculates the control ages relative to that cohort's 
    median diagnosis date, saves the result, and returns a dictionary 
    of the processed control DataFrames.

    Args:
        case_cohort_dict (dict): The dictionary of { (id, name): df }
        control_df_path (str or Path): File path to the master control 
                                       DataFrame (CSV or Pickle).
        output_folder (str): Name of the folder to save the resulting CSVs.

    Returns:
        dict: A new dictionary { (id, name): processed_control_df }
    """
    

    # --- 2. Setup Output ---
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    processed_controls_dict = {}
    print(f"Starting processing for {len(case_cohort_dict)} cohorts...")

    # --- 3. Loop Through Each Case Cohort ---
    for (concept_id, concept_name), case_df in case_cohort_dict.items():
        
        print(f"--- Processing: {concept_name} (ID: {concept_id}) ---")
        
        # Create the automated output path
        safe_filename = f"{slugify(concept_name)}_controls.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)

        try:
            # Call your function (which must be defined elsewhere in your notebook)
            processed_df = calculate_control_age_at_index_date(
                cases_df=case_df,
                controls_df=ctrl_upd, # Use the loaded master_control_df
                output_csv_path=output_csv_path,
                id_cases_col="person_id",
                case_dx_date_col="first_diag_date",
                ctrl_dob_col="date_of_birth"
            )
            
            # Store the returned DataFrame in the new dictionary
            processed_controls_dict[(concept_id, concept_name)] = processed_df
            print(f"Successfully processed and saved to {output_csv_path}\n")

        except Exception as e:
            # Catch errors (e.g., if a case_df is empty or has no dates)
            print(f"ERROR processing {concept_name}: {e}\n", file=sys.stderr)

    print("--- All cohorts processed. ---")
    return processed_controls_dict


In [ ]:
#NS cohorts
all_processed_ns_controls = process_ns_cohorts(
        case_cohort_dict=upd_ns_cases, 
        control_df_path=ctrl_upd,
        output_folder="ns_controls"
    )

In [ ]:
#NS VAX cohorts - getting controls age

import pandas as pd
import os
import re
from pathlib import Path
import sys

def slugify(text):
    """
    Converts a string into a safe component for a filename.
    'Viral disease' -> 'viral_disease'
    'Acute hepatitis C' -> 'acute_hepatitis_c'
    '...AND/OR...' -> 'and_or'
    """
    text = str(text).lower()
    # Replace non-alphanumeric/underscore with a single underscore
    text = re.sub(r'[^a-z0-9_]+', '_', text) 
    text = text.strip('_') # Clean up leading/trailing underscores
    return text

def process_ns_vax_cohorts(case_cohort_dict, 
                        control_df_path, 
                        output_folder="cohort_data_ns"):
    """
    Processes a dictionary of case cohorts against a single master control cohort.

    This function loads the master control file, then iterates through each
    case cohort, calculates the control ages relative to that cohort's 
    median diagnosis date, saves the result, and returns a dictionary 
    of the processed control DataFrames.

    Args:
        case_cohort_dict (dict): The dictionary of { (id, name): df }
        control_df_path (str or Path): File path to the master control 
                                       DataFrame (CSV or Pickle).
        output_folder (str): Name of the folder to save the resulting CSVs.

    Returns:
        dict: A new dictionary { (id, name): processed_control_df }
    """
    

    # --- 2. Setup Output ---
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    processed_controls_dict = {}
    print(f"Starting processing for {len(case_cohort_dict)} cohorts...")

    # --- 3. Loop Through Each Case Cohort ---
    for (concept_id, concept_name), case_df in case_cohort_dict.items():
        
        print(f"--- Processing: {concept_name} (ID: {concept_id}) ---")
        
        # Create the automated output path
        safe_filename = f"{slugify(concept_name)}_controls.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)

        try:
            # Call your function (which must be defined elsewhere in your notebook)
            processed_df = calculate_control_age_at_index_date(
                cases_df=case_df,
                controls_df=ctrl_upd, # Use the loaded master_control_df
                output_csv_path=output_csv_path,
                id_cases_col="person_id",
                case_dx_date_col="first_diag_date_x",
                ctrl_dob_col="date_of_birth"
            )
            
            # Store the returned DataFrame in the new dictionary
            processed_controls_dict[(concept_id, concept_name)] = processed_df
            print(f"Successfully processed and saved to {output_csv_path}\n")

        except Exception as e:
            # Catch errors (e.g., if a case_df is empty or has no dates)
            print(f"ERROR processing {concept_name}: {e}\n", file=sys.stderr)

    print("--- All cohorts processed. ---")
    return processed_controls_dict


In [ ]:
#NS vax cohorts

all_processed_ns_vax_controls = process_ns_vax_cohorts(
        case_cohort_dict=upd_ns_vax_cases, 
        control_df_path=ctrl_upd,
        output_folder="ns_vax_controls"
    )

In [ ]:
#### EXPORT raw cohort DF AS CSVs

In [ ]:
import pandas as pd
import os
import re
from pathlib import Path
import sys

def slugify(text):
    """
    Converts a string into a safe component for a filename.
    'Viral disease' -> 'viral_disease'
    'Acute hepatitis C' -> 'acute_hepatitis_c'
    '...AND/OR...' -> 'and_or'
    """
    text = str(text).lower()
    # Replace non-alphanumeric/underscore with a single underscore
    text = re.sub(r'[^a-z0-9_]+', '_', text) 
    text = text.strip('_') # Clean up leading/trailing underscores
    return text

def save_dataframes_to_csv(df_dict, 
                           output_folder="cohort_csv_output", 
                           suffix="_case_cohort"):
    """
    Saves each DataFrame in a dictionary to its own CSV file.

    The dictionary key is assumed to be a tuple: (concept_id, concept_name).
    The filename will be generated from the 'concept_name'.

    Args:
        df_dict (dict): The dictionary of { (id, name): df } to save.
                        (e.g., your 'ns_df_upd' variable)
        output_folder (str): Name of the folder to save the CSVs.
        suffix (str): The suffix to add to the filename (e.g., "_controls").
    """
    
    # 1. Create the output directory if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    print(f"Saving CSVs to folder: {output_folder}")
    
    count = 0
    # 2. Loop through the dictionary items
    for (concept_id, concept_name), df in df_dict.items():
        
        # 3. Create the automated filename
        safe_filename = f"{slugify(concept_name)}{suffix}.csv"
        output_csv_path = os.path.join(output_folder, safe_filename)
        
        try:
            # 4. Save the DataFrame to CSV
            #    index=False is generally recommended for data exports
            df.to_csv(output_csv_path, index=False)
            print(f"Successfully saved: {safe_filename}")
            count += 1
        except Exception as e:
            print(f"ERROR saving {safe_filename}: {e}", file=sys.stderr)
    
    print(f"\n--- Done. Saved {count} CSV files. ---")



In [ ]:
#NS cases
save_dataframes_to_csv(
         df_dict=upd_ns_cases, 
         output_folder="ns_cases",
         suffix="_case_cohort"
     )

In [ ]:
#NS VAX cases
save_dataframes_to_csv(
         df_dict=upd_ns_vax_cases, 
         output_folder="ns_vax_cases",
         suffix="_case_cohort"
     )

In [ ]:
#covid and flu

flu_df_upd.to_csv("ns_vax_cases/flu_case_cohort.csv")

covid_df_upd.to_csv("ns_vax_cases/covid_case_cohort.csv")